<a href="https://colab.research.google.com/github/HemaSoundarya/Global-Named-Entity-Recognition/blob/main/Global_Named_Entity_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=ce6c132434a29e907060b3d781f67fad3b823d649b7ffe15cee94299264a0345
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
import datasets
import requests

BASE_URL = "https://raw.githubusercontent.com/leondz/emerging_entities_17/master/"
TRAIN_FILE = "wnut17train.conll"
DEV_FILE = "emerging.dev.conll"
TEST_FILE = "emerging.test.annotated"

In [ ]:


def load_conll(url):
    lines = requests.get(url).text.split("\n")
    tokens, tags = [], []
    data = []

    for line in lines:
        line = line.strip()
        if line:
            token, tag = line.split("\t")
            tokens.append(token)
            tags.append(tag)
        else:
            if tokens:
                data.append({"tokens": tokens, "ner_tags": tags})
                tokens, tags = [], []

    if tokens:
        data.append({"tokens": tokens, "ner_tags": tags})

    return data

train_data = load_conll(BASE_URL + TRAIN_FILE)
dev_data = load_conll(BASE_URL + DEV_FILE)
test_data = load_conll(BASE_URL + TEST_FILE)

wnut = datasets.DatasetDict({

    "train": datasets.Dataset.from_list(train_data),
    "validation": datasets.Dataset.from_list(dev_data),
    "test": datasets.Dataset.from_list(test_data),
})


In [ ]:
from transformers import AutoTokenizer

label_list = sorted(list({tag for row in train_data for tag in row["ner_tags"]}))
label_to_id = {l: i for i, l in enumerate(label_list)}
id_to_label = {i: l for l, i in label_to_id.items()}

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(batch):
    tokenized = tokenizer(batch["tokens"], is_split_into_words=True, truncation=True)
    labels = []

    for i, word_ids in enumerate(tokenized.word_ids(batch_index=j) for j in range(len(batch["tokens"]))):
        label_ids = []
        for k, word_id in enumerate(word_ids):
            if word_id is None:
                label_ids.append(-100)
            else:
                label_ids.append(label_to_id[batch["ner_tags"][i][word_id]])
        labels.append(label_ids)

    tokenized["labels"] = labels
    return tokenized

tokenized_dataset = wnut.map(tokenize_and_align_labels, batched=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/3394 [00:00<?, ? examples/s]

Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

Map:   0%|          | 0/1287 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(batch):
    tokenized_inputs = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=128,
    )

    labels = []
    for i, word_ids in enumerate(tokenized_inputs.word_ids(batch_index=j) for j in range(len(batch["tokens"]))):
        label_ids = []
        previous_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(label_to_id[batch["ner_tags"][i][word_id]])
            else:
                label_ids.append(-100)
            previous_word_id = word_id
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = wnut.map(tokenize_and_align_labels, batched=True)


Map:   0%|          | 0/3394 [00:00<?, ? examples/s]

Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

Map:   0%|          | 0/1287 [00:00<?, ? examples/s]

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
)


In [ ]:
import requests
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Definitions needed for model, tokenizer, and label mappings
BASE_URL = "https://raw.githubusercontent.com/leondz/emerging_entities_17/master/"
TRAIN_FILE = "wnut17train.conll"

def load_conll(url):
    lines = requests.get(url).text.split("\n")
    tokens, tags = [], []
    data = []

    for line in lines:
        line = line.strip()
        if line:
            token, tag = line.split("\t")
            tokens.append(token)
            tags.append(tag)
        else:
            if tokens:
                data.append({"tokens": tokens, "ner_tags": tags})
                tokens, tags = [], []

    if tokens:
        data.append({"tokens": tokens, "ner_tags": tags})

    return data

train_data = load_conll(BASE_URL + TRAIN_FILE)

label_list = sorted(list({tag for row in train_data for tag in row["ner_tags"]}))
label_to_id = {l: i for i, l in enumerate(label_list)}
id_to_label = {i: l for l, i in label_to_id.items()}

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id_to_label,
    label2id=label_to_id
)

def predict(sentence):
    tokens = tokenizer(sentence, return_tensors="pt")
    outputs = model(**tokens)
    predictions = outputs.logits.argmax(-1).squeeze().tolist()
    ids = tokens["input_ids"].squeeze().tolist()

    for token_id, label_id in zip(ids, predictions):
        token = tokenizer.decode([token_id])
        label = id_to_label.get(label_id, "O")
        print(token, "→", label)

predict("Microsoft and OpenAI announced a strategic partnership during the AI Summit in London this March.")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[CLS] → B-creative-work
Microsoft → B-creative-work
and → B-location
Open → B-location
##A → B-location
##I → B-location
announced → B-corporation
a → B-location
strategic → B-creative-work
partnership → B-location
during → B-corporation
the → B-creative-work
AI → B-group
Summit → B-creative-work
in → B-location
London → B-creative-work
this → B-location
March → B-creative-work
. → I-product
[SEP] → I-product
